In [1]:
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)

METHOD = "hmm_pomegranate"


# 1. Read data

In [2]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/"

In [3]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = get_available_alleles(DATA_PATH)

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

ALLELES

['HLA-A*02:01']

# 2. Train HMMS
## 2.1 Code for HMM
### 2.1.1 Training code

In [4]:
from hmm_logic_methods import train_model, save_model, add_more_states_and_reset_transitions

## 2.2 Train
### 2.2.1 Train 8-mer

In [5]:
%%time
per_allele_models = dict()
per_allele_histories = dict()
TARGET_LENGTHS = [8]
for allele_name in ALLELES:
    per_kfold_per_length_data =  per_allele_per_kfold_per_length_binders_train[allele_name]
    models, histories = train_model(allele_name,per_kfold_per_length_data, TARGET_LENGTHS, verbose=True)
    per_allele_models[allele_name] = models
    per_allele_histories[allele_name] = histories
    break

split 0
train size 2392
Shuffling for fairness 
[1] Improvement: 135190.12558853778	Time (s): 0.549
[2] Improvement: 17.750770063015807	Time (s): 0.528
[3] Improvement: 19.736697401407582	Time (s): 0.954
[4] Improvement: 24.640430724830367	Time (s): 0.908
[5] Improvement: 32.03787374190142	Time (s): 0.589
[6] Improvement: 41.53275289627345	Time (s): 0.621
[7] Improvement: 52.33361080407485	Time (s): 0.82
[8] Improvement: 63.13138809717202	Time (s): 0.868
[9] Improvement: 72.46112301526591	Time (s): 0.67
[10] Improvement: 79.45645776659512	Time (s): 0.591
[11] Improvement: 84.46835606131935	Time (s): 0.862
[12] Improvement: 89.00816127424332	Time (s): 0.755
[13] Improvement: 95.04492568864953	Time (s): 1.099
[14] Improvement: 104.16143490406103	Time (s): 0.866
[15] Improvement: 116.90676931334019	Time (s): 1.057
[16] Improvement: 132.38770726056828	Time (s): 0.786
[17] Improvement: 148.51585135851928	Time (s): 0.653
[18] Improvement: 163.48912777557416	Time (s): 0.697
[19] Improvement: 

In [6]:
for allele_name in ALLELES:
    for split_num, model in per_allele_models[allele_name].items():
        history = per_allele_histories[allele_name][split_num]
        save_model(model,history, TARGET_PATH_TO_RESULTS)


In [ ]:
print("Statistics:")
num_states = [len(model.states) for model in per_allele_models[ALLELES[0]].values()]
print(f"Model states: {num_states}")
epochs = [history.epochs[-1] for history in per_allele_histories[ALLELES[0]].values()]
print(f"Epochs: {epochs}")
total_improvements = [history.total_improvement[-1] for history in per_allele_histories[ALLELES[0]].values()]
print(f"Total improvements: {total_improvements}")
total_log_probs = [history.log_probabilities[-1] for history in per_allele_histories[ALLELES[0]].values()]
print(f"Total log probs: {total_log_probs}")


total_log_probs = [history.log_probabilities[-1] for history in per_allele_histories[ALLELES[0]].values()]
print(f"Total log probs: {total_log_probs}")

sample_model = per_allele_models[ALLELES[0]][0]
sample_input = 'QLIAGQSA'
sequence = list(sample_input)
logp, path = sample_model.viterbi(sequence)
print("Viterbi for Sequence: '{}'  -- Log Probability: {} -- Path: {}".format(''.join( sequence ), logp, " ".join( state.name for idx, state in path[1:-1] ) ))
logp, path = sample_model.maximum_a_posteriori(sequence)

print("Max a post for Sequence: '{}'  -- Log Probability: {} -- Path: {}".format(''.join( sequence ), logp, " ".join( state.name for idx, state in path ) ))


### 2.2.1 Train 9-mer

In [ ]:
per_allele_9_models = dict()
per_allele_9_histories = dict()
TARGET_LENGTHS = [9]
for allele_name in ALLELES:
    per_kfold_per_length_data =  per_allele_per_kfold_per_length_binders_train[allele_name]
    models, histories = train_model(allele_name,per_kfold_per_length_data, TARGET_LENGTHS, multiple_check_input=False, n_jobs=3, verbose=True)
    per_allele_9_models[allele_name] = models
    per_allele_9_histories[allele_name] = histories


In [ ]:
for allele_name in ALLELES:
    for split_num, model in per_allele_9_models[allele_name].items():
        save_model(model, TARGET_PATH_TO_RESULTS)